# Meta-Analysis v2: Multi-Agent vs Single-Agent LLM Systems
**Author:** Xinyu Yang (r1020926), KU Leuven  
**v2 changes:** Fixes C1-C12 from peer review (see `report/review/v1-response.md`)

In [1]:
import importlib, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent))
import src.data_io, src.effect_sizes, src.pooling, src.heterogeneity
import src.moderators, src.publication_bias, src.sensitivity, src.plots, src.reporting
for mod in [src.data_io, src.effect_sizes, src.pooling, src.heterogeneity,
            src.moderators, src.publication_bias, src.sensitivity, src.plots, src.reporting]:
    importlib.reload(mod)

from src.data_io import load_verified, DATA_VERIFIED, REPORT_ASSETS
from src.effect_sizes import compute_effect_sizes
from src.pooling import random_effects
from src.heterogeneity import baujat, cooks_distance, hat_values, leave_one_out
from src.moderators import subgroup_analysis, meta_regression
from src.publication_bias import egger_test, begg_test, trim_and_fill, p_curve
from src.sensitivity import run_all_sensitivity
from src.plots import (forest_plot, forest_plot_interactive, funnel_plot,
                       baujat_plot, subgroup_forest, bubble_plot, prisma_flow)
from src.reporting import format_pooled, descriptives_table, sensitivity_table

REPORT_ASSETS.mkdir(parents=True, exist_ok=True)
print("All modules loaded (v2 — reloaded from fixed src/).")

All modules loaded (v2 — reloaded from fixed src/).


## Load & Sanity Check

In [2]:
df = load_verified()
rob = pd.read_csv(DATA_VERIFIED / "risk-of-bias.csv")
with open(Path.cwd().parent / "data" / "extracted" / "prisma-counts.json") as f:
    prisma_counts = json.load(f)

print(f"Studies loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nYear distribution:\n{df['year'].value_counts().sort_index()}")
print(f"\nArchitecture distribution:\n{df['architecture'].value_counts()}")
print(f"\nCompute parity:\n{df['compute-parity-flag'].value_counts()}")
df.head(10)

Studies loaded: 41
Columns: ['study-id', 'citation-key', 'year', 'backbone-model', 'architecture', 'n-agents', 'task-category', 'benchmark-name', 'n-items', 'n-correct-ma', 'n-correct-sa', 'compute-parity-flag', 'compute-ratio-ma-to-sa', 'source-table-or-figure', 'audit-status', 'n-items-estimated']

Year distribution:
year
2023     1
2024    11
2025    16
2026    13
Name: count, dtype: int64

Architecture distribution:
architecture
debate              20
cooperation          9
hierarchy            6
role-play            2
verifier-critic      2
planner-executor     1
moa                  1
Name: count, dtype: int64

Compute parity:
compute-parity-flag
no         25
yes         9
unclear     7
Name: count, dtype: int64


,study-id,citation-key,year,backbone-model,architecture,n-agents,task-category,benchmark-name,n-items,n-correct-ma,n-correct-sa,compute-parity-flag,compute-ratio-ma-to-sa,source-table-or-figure,audit-status,n-items-estimated
0,S001,Du2023,2023,ChatGPT (GPT-3.5),debate,3,math,GSM8K,1319,1121,1016,no,NR,Table/Fig from Du2023,unaudited,False
1,S002,Qian2024,2024,GPT-3.5-Turbo,role-play,3,code,SoftwareDev,70,1,1,no,NR,Table/Fig from Qian2024,unaudited,True
2,S003,Liang2024,2024,GPT-3.5-Turbo,cooperation,3,evaluation,Counter-Intuitive AR,200,74,56,no,NR,Table/Fig from Liang2024,unaudited,True
3,S004,Huang2024,2024,GPT-4,cooperation,3,code,HumanEval,164,158,148,no,NR,Table/Fig from Huang2024,unaudited,False
4,S005,Xue2025,2025,Gemini 2.5 Pro / GPT-5 / Grok 4 / Claude Sonnet 4,cooperation,3,other,GPQA-Diamond,198,172,170,no,NR,Table/Fig from Xue2025,verified,False
5,S006,Yang2025,2025,GPT-3.5-Turbo,cooperation,3,code,HumanEval,164,135,125,unclear,NR,Table/Fig from Yang2025,unaudited,False
6,S007,Islam2024,2024,GPT-4,planner-executor,4,code,HumanEval,164,154,145,no,NR,Table/Fig from Islam2024,verified-corrected,False
7,S008,Hong2024,2024,GPT-4,hierarchy,3,code,HumanEval,164,141,110,no,NR,Table/Fig from Hong2024,unaudited,False
8,S009,Zhang2024b,2024,mixed,cooperation,3,code,SWE-bench Lite,300,103,82,unclear,NR,Table/Fig from Zhang2024b,verified-corrected,False
9,S010,Benkovich2026,2026,GPT-5,role-play,5,code,SWE-bench 500,500,360,325,yes,NR,Table/Fig from Benkovich2026,unaudited,False


## Effect Sizes

In [3]:
df = compute_effect_sizes(df)
print(f"Effect sizes computed for {len(df)} studies")
print(f"\nLog-OR (yi) summary:\n{df['yi'].describe()}")
print(f"\nCohen's h summary:\n{df['h'].describe()}")
print(f"\nStudies with MA advantage (yi > 0): {(df['yi'] > 0).sum()}")
print(f"Studies with SA advantage (yi < 0): {(df['yi'] < 0).sum()}")
df[["study-id", "citation-key", "architecture", "benchmark-name", "yi", "sei", "h"]].round(3)

Effect sizes computed for 41 studies

Log-OR (yi) summary:
count    41.000000
mean      0.512485
std       0.795052
min      -2.296315
25%       0.194340
50%       0.364707
75%       0.702073
max       2.581899
Name: yi, dtype: float64

Cohen's h summary:
count    41.000000
mean      0.199189
std       0.342591
min      -1.077335
25%       0.096583
50%       0.155764
75%       0.246310
max       1.186914
Name: h, dtype: float64

Studies with MA advantage (yi > 0): 37
Studies with SA advantage (yi < 0): 3


,study-id,citation-key,architecture,benchmark-name,yi,sei,h
0,S001,Du2023,debate,GSM8K,0.524,0.101,0.204
1,S002,Qian2024,role-play,SoftwareDev,0.000,1.424,0.000
2,S003,Liang2024,cooperation,Counter-Intuitive AR,0.412,0.215,0.193
3,S004,Huang2024,cooperation,HumanEval,1.046,0.492,0.250
4,S005,Xue2025,cooperation,GPQA-Diamond,0.086,0.293,0.029
5,S006,Yang2025,cooperation,HumanEval,0.373,0.275,0.151
6,S007,Islam2024,planner-executor,HumanEval,0.702,0.407,0.196
7,S008,Hong2024,hierarchy,HumanEval,1.102,0.280,0.455
8,S009,Zhang2024b,cooperation,SWE-bench Lite,0.329,0.178,0.152
9,S010,Benkovich2026,role-play,SWE-bench 500,0.325,0.137,0.151


## Descriptives

In [4]:
desc = descriptives_table(df)
print("Year × Architecture × Task-Category cross-tabulation:")
desc

Year × Architecture × Task-Category cross-tabulation:


task-category           code  evaluation  factuality  general-knowledge  math  \
year  architecture                                                              
2023  debate               0           0           0                  0     1   
2024  cooperation          3           1           0                  0     0   
      debate               0           0           0                  0     0   
      hierarchy            2           0           0                  0     0   
      moa                  0           0           0                  0     0   
      planner-executor     1           0           0                  0     0   
      role-play            1           0           0                  0     0   
2025  cooperation          1           0           0                  1     0   
      debate               0           1           1                  2     2   
      hierarchy            0           0           0                  0     0   
      verifier-critic      0           0           0                  0     0   
2026  cooperation          1           0           0                  0     0   
      debate               0           0           0                  0     3   
      hierarchy            0           1           0                  0     0   
      role-play            1           0           0                  0     0   
      verifier-critic      1           0           0                  0     0   
Total                     11           3           1                  3     6   

task-category           medical  other  Total  
year  architecture                             
2023  debate                  0      0      1  
2024  cooperation             0      0      4  
      debate                  1      0      1  
      hierarchy               1      0      3  
      moa                     0      1      1  
      planner-executor        0      0      1  
      role-play               0      0      1  
2025  cooperation             0      1      3  
      debate                  1      4     11  
      hierarchy               0      1      1  
      verifier-critic         0      1      1  
2026  cooperation             0      1      2  
      debate                  3      1      7  
      hierarchy               1      0      2  
      role-play               0      0      1  
      verifier-critic         0      0      1  
Total                         7     10     41

## Primary Pooling (REML + DL Sensitivity)

In [5]:
yi, vi = df["yi"].values, df["vi"].values

res_reml = random_effects(yi, vi, method="reml")
res_dl = random_effects(yi, vi, method="dl")

print("=== REML Random-Effects ===")
print(format_pooled(res_reml))
print(f"\n=== DerSimonian-Laird (sensitivity) ===")
print(format_pooled(res_dl))

pooled = res_reml

=== REML Random-Effects ===
LOR = 0.45, 95% CI [0.25, 0.65], 95% PI [-0.74, 1.64], τ² = 0.335, I² = 96%, Q(40) = 1016.6, p = 0.0000

=== DerSimonian-Laird (sensitivity) ===
LOR = 0.41, 95% CI [0.28, 0.55], 95% PI [-0.37, 1.20], τ² = 0.145, I² = 96%, Q(40) = 1016.6, p = 0.0000


## Forest Plot

In [6]:
fig_forest = forest_plot(df.sort_values("yi"), pooled)
plt.show()
_ = forest_plot_interactive(df.sort_values("yi"), pooled)
print("Forest plots saved to report/assets/")

Forest plots saved to report/assets/


## Heterogeneity Diagnostics

In [7]:
qi, influence = baujat(yi, vi, pooled["tau2"])
fig_baujat = baujat_plot(qi, influence, df["citation-key"].tolist())
plt.show()

cooks = cooks_distance(yi, vi, pooled["tau2"])
hats = hat_values(vi, pooled["tau2"])
print(f"Top 5 Cook's distance:\n{df.iloc[np.argsort(-cooks)[:5]][['citation-key', 'yi']].assign(cooks=np.sort(cooks)[::-1][:5]).to_string()}")
print(f"\nTop 5 hat values:\n{df.iloc[np.argsort(-hats)[:5]][['citation-key', 'yi']].assign(hat=np.sort(hats)[::-1][:5]).to_string()}")

Top 5 Cook's distance:
   citation-key        yi     cooks
38     Zhou2025 -2.296315  0.290323
27       Wu2025  2.385747  0.183626
10      Xie2025  1.765138  0.112625
28     Wynn2025 -0.542617  0.088985
22   Ferber2025  2.581899  0.047873

Top 5 hat values:
   citation-key        yi       hat
21      Bao2025  0.136640  0.028647
28     Wynn2025 -0.542617  0.028635
30      Lin2025  0.452270  0.028631
37      Liu2026  0.194340  0.028570
31      Cui2025  0.364707  0.028565


## Subgroup Analysis

In [8]:
sg_task = subgroup_analysis(df, "task-category")
print("=== Subgroup by Task Category ===")
print(sg_task[["group", "k", "pooled-lor", "ci-lo", "ci-hi", "i2"]].to_string(index=False))
if "q-between" in sg_task.attrs:
    print(f"\nQ_between = {sg_task.attrs['q-between']:.2f}, df = {sg_task.attrs['df-between']}, p = {sg_task.attrs['p-between']:.4f}")
fig_sg_task = subgroup_forest(sg_task, "task-category")
plt.show()

sg_arch = subgroup_analysis(df, "architecture")
print("\n=== Subgroup by Architecture ===")
print(sg_arch[["group", "k", "pooled-lor", "ci-lo", "ci-hi", "i2"]].to_string(index=False))
if "q-between" in sg_arch.attrs:
    print(f"\nQ_between = {sg_arch.attrs['q-between']:.2f}, df = {sg_arch.attrs['df-between']}, p = {sg_arch.attrs['p-between']:.4f}")
fig_sg_arch = subgroup_forest(sg_arch, "architecture")
plt.show()

=== Subgroup by Task Category ===
            group  k  pooled-lor     ci-lo    ci-hi        i2
             code 11    0.828080  0.434897 1.221263 75.260269
       evaluation  3    0.240195 -1.077724 1.558113  0.000000
general-knowledge  3    0.015424 -3.713532 3.744379 99.662029
             math  6    0.341953  0.036455 0.647451 93.005728
          medical  7    0.420276  0.257237 0.583315 60.988876
            other 10    0.318950 -0.574947 1.212846 90.837399

Q_between = 197.56, df = 5, p = 0.0000



=== Subgroup by Architecture ===
          group  k  pooled-lor     ci-lo    ci-hi        i2
    cooperation  9    0.313409 -0.524365 1.151183 85.939667
         debate 20    0.371998  0.150299 0.593697 97.823056
      hierarchy  6    0.764568  0.021144 1.507992 79.880099
      role-play  2    0.322449  0.055566 0.589332  0.000000
verifier-critic  2    1.009143 -0.444094 2.462380 96.585329

Q_between = 32.75, df = 4, p = 0.0000


## Meta-Regression

In [9]:
mr_df = df.copy()
mr_df["year-centered"] = mr_df["year"] - mr_df["year"].mean()
mr_df["backbone-family"] = mr_df["backbone-model"].str.extract(r"(GPT|Claude|Llama|Gemini|Qwen|DeepSeek)", expand=False).fillna("other")
mr_df["n-agents"] = pd.to_numeric(mr_df["n-agents"], errors="coerce").fillna(3)
mr_df["sa-accuracy"] = mr_df["n-correct-sa"] / mr_df["n-items"]

# C8 fix: include sa-accuracy as direct frontier-paradox moderator
mr_cols = ["compute-parity-flag", "sa-accuracy", "backbone-family", "n-agents"]

mr_results = meta_regression(mr_df, mr_cols)
print("=== Meta-Regression (v2): yi ~ compute-parity + sa-accuracy + backbone + n-agents ===")
print(mr_results.round(4).to_string(index=False))
print(f"\nConditional τ² = {mr_results.attrs.get('tau2-conditional', 'N/A'):.4f}")
print(f"Unconditional τ² = {mr_results.attrs.get('tau2-unconditional', 'N/A'):.4f}")
print(f"R² = {mr_results.attrs.get('r2', 'N/A'):.1%}")
qm = mr_results.attrs.get('q-model', np.nan)
pqm = mr_results.attrs.get('p-qm', np.nan)
print(f"QM = {qm:.2f}, p = {pqm:.4f}" if not np.isnan(qm) else "QM not computed")

=== Meta-Regression (v2): yi ~ compute-parity + sa-accuracy + backbone + n-agents ===
                       coef  estimate     se       t      p  df
                  intercept    0.7766 1.0823  0.7176 0.4782  32
compute-parity-flag_unclear    0.2047 0.3345  0.6120 0.5448  32
    compute-parity-flag_yes   -0.0281 0.3246 -0.0865 0.9316  32
                sa-accuracy   -0.7695 0.5557 -1.3848 0.1757  32
        backbone-family_GPT    0.2697 0.7072  0.3814 0.7054  32
     backbone-family_Gemini    0.2966 0.7848  0.3779 0.7080  32
       backbone-family_Qwen   -0.2630 0.7704 -0.3414 0.7350  32
      backbone-family_other   -0.0700 0.7465 -0.0937 0.9259  32
                   n-agents    0.0048 0.2340  0.0206 0.9837  32

Conditional τ² = 0.3860
Unconditional τ² = 0.3351
R² = 0.0%
QM = 5.34, p = 0.7202


## Publication Bias

In [10]:
fig_funnel = funnel_plot(df, pooled)
plt.show()

egger = egger_test(yi, df["sei"].values)
print(f"Egger's test: intercept = {egger['intercept']:.3f}, t = {egger['t']:.3f}, p = {egger['p']:.4f}")

begg = begg_test(yi, vi)
print(f"Begg's test: τ = {begg['tau']:.3f}, p = {begg['p']:.4f}")

tf = trim_and_fill(yi, vi)
print(f"\nTrim-and-fill: {tf['k0']} studies imputed")
print(f"  Original pooled LOR: {tf['pooled-lor-orig']:.3f}")
print(f"  Adjusted pooled LOR: {tf['pooled-lor-adj']:.3f} [{tf['ci-lo-adj']:.3f}, {tf['ci-hi-adj']:.3f}]")

pc = p_curve(yi, df["sei"].values)
print(f"\nP-curve: {pc}")

Egger's test: intercept = 2.526, t = 31.714, p = 0.7652
Begg's test: τ = 0.056, p = 0.6054

Trim-and-fill: 0 studies imputed
  Original pooled LOR: 0.450
  Adjusted pooled LOR: 0.450 [0.251, 0.648]

P-curve: {'n-sig': 28, 'n-right-skewed': 25, 'n-total': 28, 'prop-right': np.float64(0.8928571428571429), 'binom-p': np.float64(2.744048833847046e-05), 'evidential-value': np.True_}


## Sensitivity Analyses

In [11]:
sens = run_all_sensitivity(df, rob)
print("=== Sensitivity Analyses (v2 — expanded per C7, C12) ===")
print(sensitivity_table(sens))

loo = leave_one_out(yi, vi)
loo["citation-key"] = df["citation-key"].values
print(f"\nLeave-one-out range: [{loo['mu'].min():.3f}, {loo['mu'].max():.3f}]")
print(f"Leave-one-out τ² range: [{loo['tau2'].min():.3f}, {loo['tau2'].max():.3f}]")
most_influential = loo.loc[loo["mu"].idxmin()]
print(f"Most influential (lowest pooled when removed): {most_influential['citation-key']} → LOR={most_influential['mu']:.3f}, τ²={most_influential['tau2']:.3f}")

=== Sensitivity Analyses (v2 — expanded per C7, C12) ===
| label                     |   k |   pooled-lor |   ci-lo |   ci-hi | i2   |
|:--------------------------|----:|-------------:|--------:|--------:|:-----|
| Full sample               |  41 |         0.45 |    0.25 |    0.65 | 96%  |
| Compute-parity only       |   9 |         0.31 |    0.04 |    0.57 | 85%  |
| 2025+ only                |  29 |         0.41 |    0.13 |    0.68 | 97%  |
| High-RoB removed          |  16 |         0.42 |    0.16 |    0.69 | 86%  |
| Verified studies only     |  12 |         0.45 |    0.26 |    0.64 | 73%  |
| Non-aggregated benchmarks |  36 |         0.48 |    0.24 |    0.72 | 96%  |

Leave-one-out range: [0.396, 0.476]
Leave-one-out τ² range: [0.208, 0.352]
Most influential (lowest pooled when removed): Wu2025 → LOR=0.396, τ²=0.234


## Frontier-Model Paradox

In [12]:
df["sa-accuracy"] = df["n-correct-sa"] / df["n-items"]
fig_bubble = bubble_plot(df, "sa-accuracy")
plt.show()

from scipy.stats import pearsonr
r, p = pearsonr(df["sa-accuracy"], df["yi"])
print(f"Correlation between SA baseline accuracy and MA advantage (log-OR):")
print(f"  r = {r:.3f}, p = {p:.4f}")
print(f"  Interpretation: {'Negative correlation supports frontier-model paradox' if r < 0 else 'No support for paradox in this sample'}")

Correlation between SA baseline accuracy and MA advantage (log-OR):
  r = -0.178, p = 0.2668
  Interpretation: Negative correlation supports frontier-model paradox


## PRISMA Flow + Export

In [13]:
fig_prisma = prisma_flow(prisma_counts)
plt.show()

results_summary = {
    "reml": res_reml,
    "dl": res_dl,
    "egger": egger,
    "begg": begg,
    "trim-and-fill": tf,
    "p-curve": {k: (v if not isinstance(v, (np.integer, np.floating)) else float(v)) for k, v in pc.items()},
}
with open(REPORT_ASSETS / "results-summary.json", "w") as f:
    json.dump(results_summary, f, indent=2, default=float)

desc.to_csv(REPORT_ASSETS / "descriptives.csv")
sens.to_csv(REPORT_ASSETS / "sensitivity.csv", index=False)
sg_task.to_csv(REPORT_ASSETS / "subgroup-task.csv", index=False)
sg_arch.to_csv(REPORT_ASSETS / "subgroup-arch.csv", index=False)
mr_results.to_csv(REPORT_ASSETS / "meta-regression.csv", index=False)

print(f"All outputs exported to {REPORT_ASSETS}")
print(f"Figures: {[f.name for f in REPORT_ASSETS.glob('*.png')]}")
print(f"Data: {[f.name for f in REPORT_ASSETS.glob('*.csv')]}")

All outputs exported to /Users/veto/Library/CloudStorage/OneDrive-KULeuven/Courses/Meta Analysis/Proj-Meta/report/assets
Figures: ['subgroup-task-category.png', 'funnel-plot.png', 'subgroup-architecture.png', 'forest-plot.png', 'prisma-flow.png', 'bubble-plot.png', 'baujat-plot.png']
Data: ['descriptives.csv', 'subgroup-task.csv', 'meta-regression.csv', 'subgroup-arch.csv', 'sensitivity.csv']
